# Create, Populate and Display Orders Table With Test Data (SQL)

In [5]:
%%sql
DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    amount DECIMAL(10,2),
    status VARCHAR(20)
);

INSERT INTO orders (order_id, customer_id, amount, status) VALUES
-- Customer 101: multiple paid orders + one pending
(1, 101, 25.00, 'PAID'),
(2, 101, 10.00, 'PENDING'),
(3, 101, 40.00, 'PAID'),

-- Customer 102: one paid, one cancelled
(4, 102, 15.00, 'PAID'),
(5, 102, 15.00, 'CANCELLED'),

-- Customer 103: two paid orders
(6, 103, 12.50, 'PAID'),
(7, 103, 17.50, 'PAID'),

-- Customer 104: no paid orders (edge case)
(8, 104, 50.00, 'CANCELLED'),
(9, 104, 20.00, 'PENDING'),

-- Customer 105: one paid, zero amount (edge case)
(10, 105, 0.00, 'PAID'),

-- Customer 106: large amounts (stress test)
(11, 106, 250.00, 'PAID'),
(12, 106, 175.00, 'PAID'),

-- Customer 107: mixed statuses
(13, 107, 30.00, 'PENDING'),
(14, 107, 30.00, 'PAID');

-- Customer 108: no orders at all (implicit edge case)
-- (not represented in table)
-- This tests whether your Python logic handles missing keys gracefully.

SELECT order_id, customer_id, amount, status
FROM orders;

order_id,customer_id,amount,status
1,101,25.00,PAID
2,101,10.00,PENDING
3,101,40.00,PAID
4,102,15.00,PAID
5,102,15.00,CANCELLED
6,103,12.50,PAID
7,103,17.50,PAID
8,104,50.00,CANCELLED
9,104,20.00,PENDING
10,105,0.00,PAID


# Create, Populate and Display Python Dictionaries List With Test Data (Python)

In [3]:
orders = [
    # Customer 101: multiple paid orders + one pending
    {"order_id": 1, "customer_id": 101, "amount": 25.00, "status": "PAID"},
    {"order_id": 2, "customer_id": 101, "amount": 10.00, "status": "PENDING"},
    {"order_id": 3, "customer_id": 101, "amount": 40.00, "status": "PAID"},

    # Customer 102: one paid, one cancelled
    {"order_id": 4, "customer_id": 102, "amount": 15.00, "status": "PAID"},
    {"order_id": 5, "customer_id": 102, "amount": 15.00, "status": "CANCELLED"},

    # Customer 103: two paid orders
    {"order_id": 6, "customer_id": 103, "amount": 12.50, "status": "PAID"},
    {"order_id": 7, "customer_id": 103, "amount": 17.50, "status": "PAID"},

    # Customer 104: no paid orders
    {"order_id": 8, "customer_id": 104, "amount": 50.00, "status": "CANCELLED"},
    {"order_id": 9, "customer_id": 104, "amount": 20.00, "status": "PENDING"},

    # Customer 105: one paid, zero amount
    {"order_id": 10, "customer_id": 105, "amount": 0.00, "status": "PAID"},

    # Customer 106: large amounts (stress test)
    {"order_id": 11, "customer_id": 106, "amount": 250.00, "status": "PAID"},
    {"order_id": 12, "customer_id": 106, "amount": 175.00, "status": "PAID"},

    # Customer 107: mixed statuses
    {"order_id": 13, "customer_id": 107, "amount": 30.00, "status": "PENDING"},
    {"order_id": 14, "customer_id": 107, "amount": 30.00, "status": "PAID"},
]
def print_orders_table(orders):
    # Determine column widths
    headers = ["order_id", "customer_id", "amount", "status"]
    col_widths = {h: len(h) for h in headers}

    for order in orders:
        for h in headers:
            col_widths[h] = max(col_widths[h], len(str(order[h])))

    # Build format string
    row_format = " | ".join(f"{{:{col_widths[h]}}}" for h in headers)

    # Print header
    print(row_format.format(*headers))
    print("-" * (sum(col_widths.values()) + 3 * (len(headers) - 1)))

    # Print rows
    for order in orders:
        print(row_format.format(
            order["order_id"],
            order["customer_id"],
            f"{order['amount']:.2f}",
            order["status"]
        ))

# Call it
print_orders_table(orders)



order_id | customer_id | amount | status   
-------------------------------------------
       1 |         101 | 25.00  | PAID     
       2 |         101 | 10.00  | PENDING  
       3 |         101 | 40.00  | PAID     
       4 |         102 | 15.00  | PAID     
       5 |         102 | 15.00  | CANCELLED
       6 |         103 | 12.50  | PAID     
       7 |         103 | 17.50  | PAID     
       8 |         104 | 50.00  | CANCELLED
       9 |         104 | 20.00  | PENDING  
      10 |         105 | 0.00   | PAID     
      11 |         106 | 250.00 | PAID     
      12 |         106 | 175.00 | PAID     
      13 |         107 | 30.00  | PENDING  
      14 |         107 | 30.00  | PAID     


# Model A SQL

In [2]:
%%sql
SELECT customer_id, SUM(amount) AS total
FROM orders
WHERE status = 'PAID'
GROUP BY customer_id;

customer_id,total
101,65.00
102,15.00
103,30.00
105,0.00
106,425.00
107,30.00


# Model A Python

In [27]:
def total_paid(orders):
    totals = {}
    for o in orders:
        if o["status"] == "PAID":
            totals[o["customer_id"]] = totals.get(o["customer_id"], 0) + o["amount"]
    return totals
# Call it
total_paid(orders)

{101: 65.0, 102: 15.0, 103: 30.0, 105: 0.0, 106: 425.0, 107: 30.0}

# Model B SQL

In [3]:
%%sql
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
WHERE status != 'CANCELLED'
GROUP BY customer_id
ORDER BY total_spent DESC;

customer_id,total_spent
106,425.00
101,75.00
107,60.00
103,30.00
104,20.00
102,15.00
105,0.00


# Model B Python

In [ ]:
totals = {}

for order in orders:
    if order["status"] != "CANCELLED":
        cid = order["customer_id"]
        totals[cid] = totals.get(cid, 0) + order["amount"]

sorted_totals = sorted(totals.items(), key=lambda x: x[1], reverse=True)
# Call it
sorted_totals

[(106, 425.0),
 (101, 75.0),
 (107, 60.0),
 (103, 30.0),
 (104, 20.0),
 (102, 15.0),
 (105, 0.0)]

# Model C SQL

In [4]:
%%sql
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
WHERE status = 'PAID'
GROUP BY customer_id
HAVING SUM(amount) > 0
ORDER BY total_spent DESC;

customer_id,total_spent
106,425.00
101,65.00
103,30.00
107,30.00
102,15.00


# Model C Python

In [30]:
from collections import defaultdict

totals = defaultdict(float)

for order in orders:
    if order["status"] == "PAID":
        totals[order["customer_id"]] += order["amount"]

result = sorted(
    [(cid, amt) for cid, amt in totals.items() if amt > 0],
    key=lambda x: x[1],
    reverse=True
)
# Call it
result

[(106, 425.0), (101, 65.0), (103, 30.0), (107, 30.0), (102, 15.0)]

# My SQL

In [ ]:
%%sql
SELECT customer_id, SUM(CASE WHEN status ILIKE 'PAID' AND amount >= 0 THEN amount ELSE 0 END) AS total
-- ensures all customers are included, demonstrates excluding refunds and handles lower/mixed case status entries
FROM orders
GROUP BY customer_id;

customer_id,total
101,65.00
103,30.00
104,0
105,0.00
107,30.00
102,15.00
106,425.00


# My Python

In [ ]:
def total_spent(orders_dict_list):
    output_list = []
    # Create list of all customers
    customers=set()
    for order in orders_dict_list:
        # Create new customer entry
        if order["customer_id"] not in customers:
            output_list.append({"customer_id" : order["customer_id"], "total_spent" : 0})
            customers.add(order["customer_id"])
        # Add order amount to total
        if order["status"].upper() == 'PAID'if order["status"].upper() == 'PAID' and order["amount"] > 0::
            # Find customer in output list
            customer_index = next((index for index, customer in enumerate(output_list) if customer["customer_id"] == order["customer_id"]), None)
            # Add order amount to total spent
            output_list[customer_index]["total_spent"] += order["amount"]
    return output_list
# Call it
total_spent(orders)

[{'customer_id': 101, 'total_spent': 65.0},
 {'customer_id': 102, 'total_spent': 15.0},
 {'customer_id': 103, 'total_spent': 30.0},
 {'customer_id': 104, 'total_spent': 0},
 {'customer_id': 105, 'total_spent': 0},
 {'customer_id': 106, 'total_spent': 425.0},
 {'customer_id': 107, 'total_spent': 30.0}]